In [70]:
import openai
from qdrant_client import QdrantClient
from langsmith import traceable, get_current_run_tree

In [71]:
from dotenv import load_dotenv
import os

load_dotenv("../../.env", override=True)

True

### Embedding function

In [72]:
@traceable(
    name="embed_query",
    run_type="embedding",
    metadata={
        "ls_provider": "openai",
        "ls_model_name": "text-embedding-3-small"

    }  
)
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )

    current_run = get_current_run_tree()
    if current_run:
        current_run.metadata["usage_metadata"] = {
            "input_tokens": response.usage.prompt_tokens,
            "total_tokens": response.usage.total_tokens,

        }

    return response.data[0].embedding

### Retrieval Function

In [73]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [74]:
@traceable(
    name="retrieve_data",
    run_type="retriever"   
)
def retrieve_data(query, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01",
        query=query_embedding,
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocessed_description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }

### Format retrieved context function

In [75]:
@traceable(
    name="format_retrieved_context",
    run_type="prompt"   
)
def process_context(context):

    formatted_context = ""

    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return formatted_context

### Create prompt template function

In [76]:
@traceable(
    name="build_prompt",
    run_type="prompt"   
)
def build_prompt(preprocessed_context, question):

    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
{preprocessed_context}

Question:
{question}    
"""

    return prompt

### Generate answer function

In [77]:
@traceable(
    name="generate_answer",
    run_type="llm", 
    metadata={
        "ls_provider": "openai",
        "ls_model_name": "gpt-5.4-nano"

    }  
)
def generate_answer(prompt):

    response = openai.chat.completions.create(
        model="gpt-5.4-nano",
        messages=[
            {"role": "system", "content": prompt}
        ],
        reasoning_effort="none"
    )

    current_run = get_current_run_tree()
    if current_run:
        current_run.metadata["usage_metadata"] = {
            "input_tokens": response.usage.prompt_tokens,
            "output_tokens": response.usage.completion_tokens,
            "total_tokens": response.usage.total_tokens,

        }

    return response.choices[0].message.content

### Combined RAG pipeline

In [78]:
@traceable(
    name="rag_pipeline",   
)
def rag_pipeline(question, top_k=5):

    retrieved_context = retrieve_data(question, k=top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)

    return answer

In [79]:
print(rag_pipeline("Do you have a USB connectable fan for hot summers?"))

Yes. We have a USB-powered fan that’s made for cooling in hot weather:

Marame 120mm 5v USB Powered Fan with Speed Controller (ID: B0BRJS644Z)  
- USB-powered (3.3 ft USB cable)  
- Speed control switch with off/low/medium/high  
- Designed to help prevent devices from overheating (good for tight electronics areas too)

We also have a smaller option:
HZD Desk Fan Rechargeable / Mini Portable Fan (ID: B0BXC72RLD)  
- USB-powered (4.9 ft USB cable) with low/medium/high  
- Quiet operation (noise less than 50 dB as listed)  
- Note: it says it does not come with a battery

Which size do you want: a larger 120mm fan, or a small personal desk fan?


In [13]:
print(rag_pipeline("Could you suggest me some earphones? I am only interested in the ones that have above 4 rating.", 10))

Sure. Here are the earphones in the available products list with a rating above 4.0:

1) B0CH8DRD6K (rating 4.9) – Bluetooth 5.3 sports headband headphones with built-in ENC mic, for workouts/calling/gaming, 20+ hours playtime.

2) B0BRV544MV (rating 4.4) – Weekly Bluetooth 5.3 wireless earbuds with charging case, 40H cycle playtime, deep bass, 4-mic clear calls, IPX7 waterproof.

3) B0BBF2VC6X (rating 4.2) – Volkano Ninja kids wired earphones (3.5mm) with microphone, volume-limited to 85dB, includes a carry case.

4) B09X9838WY (rating 4.2) – Jesebang Bluetooth 5.2 wireless earbuds with charging case, built-in mic, 40H total playtime, Type-C quick charge, IP7 waterproof.

5) B0CFHWF326 (rating 4.1) – MUSICOZY Bluetooth 5.3 headband headphones with ENC mic, designed for sports/workout calling.

If you tell me whether you want wired or wireless (and for phone calls vs workouts vs sleep), I can narrow it to the best 1-2 choices.


In [26]:
print(rag_pipeline("Could you suggest me some earphones? I am only interested in the ones that have bellow 4 rating.", 10))

Sure—here are earphones/headphones from the available products that have a rating below 4.0:

1) pamu S29 Wireless Earbuds (Bluetooth 5.2, ANC/ENC, USB-C fast charge) — rating 3.9  
2) RUNAR RNR1 Wireless Neckband Running Headphones (Bluetooth 5.0, sweatproof, TF/SD compatible) — rating 3.8  

If you tell me what you need most (sports/waterproof, best calls, active noise cancellation, or wired vs wireless), I can help you pick between these two.
